# Lecture 11 Lab — Attention: Finding the Needle in a Long Sequence
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, matplotlib, PyTorch |
| **Hardware** | CPU is fine — the core experiments take two to three minutes |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. The Problem

A robot joint occasionally takes an **impact** — a collision with a fixture, a jammed part, a hard inclusion in the workpiece. The impact itself is brief: four samples of torque transient in a telemetry stream thousands of samples long.

How much damage it causes depends on two things measured **at that same instant**:

- how hard the impact was, and
- how heavily the joint was already loaded when it happened.

An impact at light load is absorbed. The same impact at near-maximum load is what cracks a gearbox tooth.

> **Your task:** given a long window of 3-channel joint telemetry, predict the resulting degradation.

### Why this task is hard for the models you already know

The informative event sits at an arbitrary position `p` in the window. To answer, a model must *find* that position, then read **two different channels** there and combine them.

Week 10's LSTM processes the sequence one step at a time, carrying everything it needs in a fixed-size hidden state. Whatever it learns at step `p` must survive all the way to step `L-1`.

Attention takes a different approach: it lets the output look **directly** at any position, however far back. Today you will build that mechanism from scratch and measure exactly when it starts to matter.

### Learning objectives

- Implement scaled dot-product attention from first principles and verify it against PyTorch's own implementation.
- Build an attention-based model and explain the role of queries, keys and values.
- Measure how sequence length changes the LSTM-versus-attention comparison, at a fixed training budget.
- Interpret attention weights, and use them to check *what* the model is looking at.
- Explain where positional information matters and where it does not.

## 2. Setup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_absolute_error

torch.manual_seed(s_id)
np.random.seed(s_id)
torch.set_num_threads(2)
print("PyTorch", torch.__version__)

## 3. Part A — The Telemetry

Three channels per time step:

| Channel | What it is |
|---|---|
| `torque` | Joint torque. Tracks load, plus the brief impact transient. |
| `load` | Slowly varying mechanical load on the joint. |
| `temperature` | Drifts randomly. **A pure distractor** — it tells you nothing. |

The target is `impact_magnitude × (1 + 0.8 × load_at_impact)`. Note that it does **not** depend on *where* the impact happened — only on what was happening there. Keep that in mind for §9.

In [ ]:
N_CH = 3
CH_NAMES = ["torque", "load", "temperature"]


def make_dataset(n, seed, seq_len):
    """Long joint-telemetry windows, each containing one brief impact."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n, seq_len, N_CH), dtype=np.float32)
    y = np.zeros(n, dtype=np.float32)
    impact_pos = np.zeros(n, dtype=np.int64)

    for i in range(n):
        t = np.linspace(0, 1, seq_len)
        load = 0.5 + 0.35 * np.sin(2 * np.pi * rng.uniform(0.5, 1.5) * t + rng.uniform(0, 6.3))
        load += rng.normal(0, 0.02, seq_len)
        temperature = np.cumsum(rng.normal(0, 0.03, seq_len)) + rng.uniform(-0.5, 0.5)
        torque = 0.8 * load + rng.normal(0, 0.05, seq_len)

        p = int(rng.integers(5, seq_len - 5))                 # where the impact lands
        magnitude = rng.uniform(0.6, 2.0)
        transient = np.array([0.45, 1.0, 0.6, 0.25])          # 4-sample shape
        torque[p:p + 4] += magnitude * transient[: len(torque[p:p + 4])]

        X[i, :, 0], X[i, :, 1], X[i, :, 2] = torque, load, temperature
        y[i] = magnitude * (1.0 + 0.8 * load[p]) + rng.normal(0, 0.03)
        impact_pos[i] = p
    return X, y, impact_pos


def prepare(seq_len, n_train=2000, n_test=800):
    Xtr, ytr, _ = make_dataset(n_train, 0, seq_len)
    Xte, yte, pos = make_dataset(n_test, 1, seq_len)
    mu = Xtr.reshape(-1, N_CH).mean(0)
    sd = Xtr.reshape(-1, N_CH).std(0)
    return (((Xtr - mu) / sd).astype(np.float32), ytr,
            ((Xte - mu) / sd).astype(np.float32), yte, pos)


Xd, yd, posd = make_dataset(3, s_id, 300)
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
for c in range(N_CH):
    axes[c].plot(Xd[0, :, c], color=["#1B2A41", "#3E5C76", "#8A97A8"][c], linewidth=1.0)
    axes[c].set_ylabel(CH_NAMES[c])
    axes[c].axvline(posd[0], color="#FF6A39", linestyle="--", linewidth=1.2)
axes[0].set_title(f"One 300-step window. The impact is at step {posd[0]} (orange line); target = {yd[0]:.2f}")
axes[2].set_xlabel("time step")
plt.tight_layout(); plt.show()
print("Look at the torque channel: the whole answer lives in four of those 300 samples.")


## 4. Part B — Scaled Dot-Product Attention, From Scratch

This is the one equation at the heart of every transformer:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

Read it as a soft lookup table:

- **Query (Q)** — what I am looking for.
- **Keys (K)** — a label advertised by each position.
- **Values (V)** — the content each position offers.

The dot product of the query with each key scores how well that position matches. A softmax turns the scores into weights summing to 1, and the output is the weighted average of the values.

**Why divide by √d_k?** With `d_k` large, dot products grow in magnitude, pushing the softmax into a region where it is nearly one-hot and its gradient nearly zero. Dividing by √d_k keeps the scores in a sensible range. §10 asks you to test this.

Fill in the `TODO`.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Q: (batch, n_queries, d_k)
    K: (batch, n_keys,    d_k)
    V: (batch, n_keys,    d_v)
    returns: output (batch, n_queries, d_v), weights (batch, n_queries, n_keys)
    """
    d_k = Q.shape[-1]

    # TODO (3 lines):
    #  1. scores  = Q @ K-transposed, divided by sqrt(d_k)
    #     (transpose the LAST TWO dims of K, not all of them: K.transpose(-2, -1))
    #  2. weights = softmax of scores over the KEYS axis (dim=-1)
    #  3. output  = weights @ V
    scores = None
    weights = None
    output = None
      

    return output, weights

In [ ]:
# Verify against PyTorch's own implementation before trusting it
torch.manual_seed(s_id)
Q = torch.randn(4, 2, 16)
K = torch.randn(4, 9, 16)
V = torch.randn(4, 9, 16)

mine, w = scaled_dot_product_attention(Q, K, V)
assert mine is not None and w is not None, "scaled_dot_product_attention still returns None - fill in the TODO above"
reference = F.scaled_dot_product_attention(Q, K, V)

print("Student ID : "+str(s_id)) # don't chage this code!
print("output shape :", tuple(mine.shape), " (expect (4, 2, 16))")
print("weights shape:", tuple(w.shape), "   (expect (4, 2, 9))")
print("each query's weights sum to 1:", torch.allclose(w.sum(-1), torch.ones(4, 2), atol=1e-5))
print("matches torch.nn.functional :", torch.allclose(mine, reference, atol=1e-5))
assert torch.allclose(mine, reference, atol=1e-5), "Your implementation does not match PyTorch's."
print("\nVerified.")

## 5. Part C — An Attention-Based Model

The simplest model that uses attention: project each time step into a `d`-dimensional space, then let **one learnable query** attend over all positions and pool them into a single context vector.

That query is the model's learned question — something like *"where is the impact?"* — and because there is exactly one of them, we get exactly one attention weight per time step. That makes the model unusually easy to inspect, which we exploit in §9.

Fill in the `TODO`.

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, n_channels=N_CH, d=48):
        super().__init__()
        self.proj = nn.Linear(n_channels, d)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.1)   # the learned question
        self.to_k = nn.Linear(d, d)
        self.to_v = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x, return_weights=False):
        h = torch.relu(self.proj(x))                       # (B, L, d)
        Q = self.query.expand(x.shape[0], -1, -1)          # (B, 1, d)

        # TODO: build the keys and values from h using self.to_k and self.to_v,
        # then call your scaled_dot_product_attention(Q, K, V).
        K = None
        V = None
        context, weights = None, None

        assert context is not None, "AttentionPool.forward still has TODOs - fill them in"
        out = self.head(context.squeeze(1))                # (B, 1)
        return (out, weights.squeeze(1)) if return_weights else out


class LSTMBaseline(nn.Module):
    """Week 10's model, unchanged."""
    def __init__(self, n_channels=N_CH, hidden=96):
        super().__init__()
        self.lstm = nn.LSTM(n_channels, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

print("Student ID : "+str(s_id)) # don't chage this code!
probe = AttentionPool()
print(probe)
print("\nAttentionPool parameters:", sum(p.numel() for p in probe.parameters() if p.requires_grad))
print("LSTMBaseline parameters  :", sum(p.numel() for p in LSTMBaseline().parameters() if p.requires_grad))

## 6. Part D — Training Helpers

In [ ]:
def train_model(model, X, y, epochs=40, lr=2e-3, batch_size=64, seed=0):
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1)),
                        batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
    return model


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X)).squeeze(1).numpy()


def run_one(model_fn, seq_len, epochs=40):
    """Train one model on one sequence length; return (MAE, seconds)."""
    Xtr, ytr, Xte, yte, _ = prepare(seq_len)
    t0 = time.time()
    model = train_model(model_fn(), Xtr, ytr, epochs=epochs)
    return mean_absolute_error(yte, predict(model, Xte)), time.time() - t0

## 7. Part E — The Experiment: Does Sequence Length Change the Answer?

Both models get **exactly the same training budget** (40 epochs, same data, same optimizer). The only thing we vary is how long the sequence is.

This takes roughly two minutes.

In [ ]:
LENGTHS = [60, 300]
results = {}
print("Student ID : "+str(s_id)) # don't chage this code!
for seq_len in LENGTHS:
    mae_a, t_a = run_one(AttentionPool, seq_len)
    mae_l, t_l = run_one(LSTMBaseline, seq_len)
    results[seq_len] = dict(attn_mae=mae_a, attn_s=t_a, lstm_mae=mae_l, lstm_s=t_l)
    print(f"L={seq_len:>4}   attention MAE {mae_a:.4f} ({t_a:>5.0f}s)   "
          f"LSTM MAE {mae_l:.4f} ({t_l:>5.0f}s)")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
xs = np.arange(len(LENGTHS)); w = 0.36
axes[0].bar(xs - w/2, [results[L]["lstm_mae"] for L in LENGTHS], w, color="#3E5C76", label="LSTM")
axes[0].bar(xs + w/2, [results[L]["attn_mae"] for L in LENGTHS], w, color="#FF6A39", label="Attention")
axes[0].set_xticks(xs); axes[0].set_xticklabels([f"L = {L}" for L in LENGTHS])
axes[0].set_ylabel("test MAE"); axes[0].set_title("Accuracy at a fixed 40-epoch budget"); axes[0].legend()

axes[1].bar(xs - w/2, [results[L]["lstm_s"] for L in LENGTHS], w, color="#3E5C76", label="LSTM")
axes[1].bar(xs + w/2, [results[L]["attn_s"] for L in LENGTHS], w, color="#FF6A39", label="Attention")
axes[1].set_xticks(xs); axes[1].set_xticklabels([f"L = {L}" for L in LENGTHS])
axes[1].set_ylabel("training time (s)"); axes[1].set_title("Cost of those same 40 epochs"); axes[1].legend()
plt.tight_layout(); plt.show()

> **Read the short sequence first.** At `L = 60` the two models should be close to tied. Attention is not magic — on a short window the LSTM's hidden state has no trouble carrying what it saw.
>
> At `L = 300` the picture changes completely, and so does the training time. The LSTM must take 300 sequential steps before it can produce a single prediction; attention scores all 300 positions in one parallel operation.

## 8. Part F — Is the LSTM Simply Under-Trained?

A fair objection: maybe the LSTM just needs longer. It is a real possibility and worth testing rather than assuming.

Set the flag below to `True` to run it (about three minutes on two cores). The measured result on the reference machine was:

| L = 300 | MAE | training time |
|---|---|---|
| Attention, 40 epochs | 0.063 | 11 s |
| LSTM, 40 epochs | 0.538 | 80 s |
| **LSTM, 100 epochs** | **0.223** | **190 s** |

So more training *does* help — the LSTM is not incapable. But after 2.5× the epochs and 17× attention's training time, it is still roughly 3.5× worse. That is the honest version of the claim: not "the LSTM cannot do this", but "it costs disproportionately more to get less".

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
RUN_LONG_LSTM = True      # set True to verify the table above yourself (~3 min)

if RUN_LONG_LSTM:
    mae_long, t_long = run_one(LSTMBaseline, 300, epochs=100)
    print(f"LSTM, 100 epochs at L=300: MAE {mae_long:.4f}  ({t_long:.0f}s)")
else:
    print("Skipped. Set RUN_LONG_LSTM = True to run it.")

## 9. Part G — What Is the Model Looking At?

Because our model has a single query, it produces exactly one attention weight per time step. Plot that and you can see, directly, which part of the sequence the model used.

Fill in the `TODO`.

In [ ]:
SEQ = 300
Xtr, ytr, Xte, yte, impact_pos = prepare(SEQ)
attn_model = train_model(AttentionPool(), Xtr, ytr, epochs=40)
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"attention model MAE at L={SEQ}: {mean_absolute_error(yte, predict(attn_model, Xte)):.4f}")

attn_model.eval()
with torch.no_grad():
    # TODO: run the model on the first 400 test windows with return_weights=True,
    # and keep the weights. Convert them to a NumPy array called `weights`.
    # Hint: the forward method returns (output, weights) when return_weights=True.
    weights = None



print("weights shape:", weights.shape, f" (expect (400, {SEQ}))")
assert np.allclose(weights.sum(axis=1), 1.0, atol=1e-4), "each row of attention weights should sum to 1"

In [ ]:
peak = weights.argmax(axis=1)
true_pos = impact_pos[:len(weights)]
within_3 = np.abs(peak - true_pos) <= 3
print(f"attention peak within 3 steps of the real impact: {within_3.mean()*100:.1f}% of windows")
print("Nobody told the model where the impact was. It was never given the position as a label.")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6.5))
for row, i in enumerate([0, 1, 2]):
    axes[row].plot(weights[i], color="#FF6A39", linewidth=1.1, label="attention weight")
    axes[row].axvline(true_pos[i], color="#1B2A41", linestyle="--", linewidth=1.3, label="true impact")
    axes[row].set_ylabel("weight")
    axes[row].legend(fontsize=8, loc="upper right")
axes[0].set_title("Attention over the sequence, three test windows")
axes[2].set_xlabel("time step")
plt.tight_layout(); plt.show()

> This is the payoff of an interpretable mechanism. In Week 9 you looked at CNN feature maps to see *what patterns* the network had learned. Here you can see *where in the input* it chose to look — and verify that against ground truth you never trained on.
>
> For a safety-relevant system that matters: a model that predicts damage correctly **and** can point at the event responsible is far easier to trust, and to debug when it is wrong.

## 9b. Optional — A Pretrained Vision Transformer

A Vision Transformer applies exactly the mechanism you just built, with one change: instead of time steps, the sequence is **image patches**. A 224×224 image is cut into 196 patches of 16×16 pixels, each patch is projected into a token, and self-attention runs over those 196 tokens.

The cell below loads a pretrained ViT and reports its structure. **It needs to download about 330 MB of weights**, so it will fail on a restricted network — that is expected, and nothing later depends on it. The concept is covered in the lecture either way.

In [ ]:
TRY_PRETRAINED_VIT = False     # set True if your network allows model downloads

if TRY_PRETRAINED_VIT:
    try:
        from torchvision.models import vit_b_16, ViT_B_16_Weights
        vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        vit.eval()
        n_params = sum(p.numel() for p in vit.parameters())
        print(f"ViT-B/16 loaded: {n_params/1e6:.0f}M parameters")
        print(f"patch size      : {vit.patch_size}x{vit.patch_size}")
        print(f"encoder blocks  : {len(vit.encoder.layers)}")
        print(f"attention heads : {vit.encoder.layers[0].num_heads} per block")
        print(f"\ntokens per image: (224/{vit.patch_size})^2 = "
              f"{(224//vit.patch_size)**2} patches, plus 1 class token")
        print("Each block runs the same scaled dot-product attention you implemented in Part B.")
    except Exception as e:
        print(f"Could not load pretrained weights ({type(e).__name__}).")
        print("This needs internet access to the PyTorch model host - skip it and move on.")
else:
    print("Skipped. Set TRY_PRETRAINED_VIT = True to try it (needs ~330 MB download).")

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. At `L = 60` the two models were close to tied, and at `L = 300` they were not. Explain the mechanism behind that difference, in terms of how far information has to travel inside each model.**

*Your answer:*

**2. Our attention model has no positional encoding at all — it does not know the order of the time steps. Explain why it can still solve this particular task, and give one modification to the target that would suddenly make positional information essential.**

*Your answer:*

**3. Delete the `/ sqrt(d_k)` from your attention function and re-run the §4 verification cell, then re-train the model in §9. Describe what changes, and explain it in terms of what large dot products do to a softmax.**

*Your answer:*

**4. The temperature channel is pure noise, by construction. Using the attention weights, describe how you could check whether the model learned to ignore it. (You do not have to run this — describe the experiment.)**

*Your answer:*

**5. Attention compares every query against every key. With one query that is cheap, but a full transformer uses one query *per position*. Work out how the number of comparisons grows with sequence length `L`, and explain why that is the central obstacle to feeding very long sensor logs into a transformer.**

*Your answer:*

**6. You have now seen four ways of building structure into a model: fixed features (Weeks 2–6), convolution (Week 9), recurrence (Week 10) and attention (Week 11). For your own semester project, which one matches the structure of your data, and why?**

*Your answer:*

## 11. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The example telemetry plot (§3), the verification output (§4), the length-comparison charts (§7), the localization percentage and attention plots (§9).
- Written answers to the six reflection questions.


Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| `scaled_dot_product_attention` correct and verified against PyTorch | 25% |
| `AttentionPool.forward` correctly wired (Q, K, V) | 20% |
| Length experiment run and interpreted | 20% |
| Attention weights extracted, localization measured and plotted | 20% |
| Reflection questions | 10% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 5% |

**Note :** Actual weights are subjected to Grader's discretion.

---
**Next week:** *Reinforcement Learning* — no labelled answers at all. A policy that learns which action to take from reward alone.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>